In [2]:
import sys
sys.path.append("../")
from model_2 import core_model
import safetensors

In [4]:
import torch

In [5]:
state_dict = torch.load("../models/nyx_2.pth", map_location="cpu")

In [6]:
state_dict

OrderedDict([('model.embed_tokens.weight',
              tensor([[ 0.0513,  0.0073,  0.0046,  ..., -0.0163,  0.0274, -0.0141],
                      [ 0.0641, -0.0422,  0.0149,  ...,  0.0266,  0.0255, -0.0027],
                      [ 0.0237,  0.0062, -0.0126,  ...,  0.0218,  0.0025, -0.0029],
                      ...,
                      [-0.0524,  0.0051,  0.0002,  ...,  0.0003,  0.0004,  0.0088],
                      [-0.0524,  0.0051,  0.0002,  ...,  0.0003,  0.0004,  0.0088],
                      [-0.0524,  0.0051,  0.0002,  ...,  0.0003,  0.0004,  0.0088]])),
             ('model.layers.0.self_attn.q_proj.weight',
              tensor([[ 0.1856, -0.1203, -0.0804,  ..., -0.0383,  0.0860,  0.0868],
                      [ 0.0707, -0.1963, -0.1221,  ...,  0.0545,  0.0653,  0.0756],
                      [ 0.0420, -0.2312, -0.1442,  ...,  0.0269,  0.0666,  0.0791],
                      ...,
                      [ 0.0841, -0.0418, -0.1488,  ..., -0.0280,  0.1374, -0.0583],
    

In [7]:
core_model.load_state_dict(state_dict)

<All keys matched successfully>

In [8]:
from safetensors.torch import save_file


In [9]:
safetensors_state_dict = core_model.state_dict()

In [10]:
save_file(safetensors_state_dict, "../models/nyx.safetensors")

In [11]:
from transformers import PretrainedConfig


In [14]:
class CoreOutlineConfig(PretrainedConfig):
    """Configuration class for CoreOutline Qwen model."""

    def __init__(
        self,
        vocab_size=151936,
        hidden_size=1024,
        intermediate_size=2816,
        num_hidden_layers=24,
        num_attention_heads=16,
        num_key_value_heads=16,
        max_position_embeddings=32768,
        initializer_range=0.02,
        rms_norm_eps=1e-6,
        use_cache=True,
        pad_token_id=151643,
        bos_token_id=151643,
        eos_token_id=151645,
        tie_word_embeddings=False,
        rope_theta=1000000.0,
        use_sliding_window=False,
        sliding_window=None,
        max_window_layers=28,
        output_attentions=False,
        output_hidden_states=False,
        use_return_dict=True,
        gradient_checkpointing=False,
        **kwargs,
    ):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.use_cache = use_cache
        self.pad_token_id = pad_token_id
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.tie_word_embeddings = tie_word_embeddings
        self.rope_theta = rope_theta
        self.use_sliding_window = use_sliding_window
        self.sliding_window = sliding_window
        self.max_window_layers = max_window_layers
        self.output_attentions = output_attentions
        self.output_hidden_states = output_hidden_states
        self.gradient_checkpointing = gradient_checkpointing

    def to_dict(self):
        return {
            key: getattr(self, key)
            for key in self.__dict__
            if not key.startswith("_")  # Skip private attributes
        }

In [15]:
config = CoreOutlineConfig()

In [16]:
config.save_pretrained("../models/Nyx")

In [17]:
from transformers import AutoTokenizer

In [18]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B")

In [19]:
inputs = tokenizer("Input text", return_tensors="pt")

In [21]:
inputs['input_ids'].shape

torch.Size([1, 2])

In [30]:
dummy_input = torch.randn(1, 12)  # Change shape to your model’s input
torch.onnx.export(
    core_model, 
    inputs['input_ids'], 
    "model.onnx", 
    export_params=True, 
    opset_version=20, 
    do_constant_folding=True, 
    input_names=["input"], 
    output_names=["output"], 
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)


UnsupportedOperatorError: Exporting the operator 'aten::rms_norm' to ONNX opset version 20 is not supported. Please feel free to request support or submit a pull request on PyTorch GitHub: https://github.com/pytorch/pytorch/issues.

In [31]:
print(torch.__config__.show())


PyTorch built with:
  - C++ Version: 201703
  - MSVC 192930157
  - Intel(R) oneAPI Math Kernel Library Version 2025.0.1-Product Build 20241031 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.5.3 (Git Hash 66f0cb9eb66affd2da3bf5f8d897376f04aae6af)
  - OpenMP 2019
  - LAPACK is enabled (usually provided by MKL)
  - CPU capability usage: AVX2
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;arch=compute_37,code=sm_37;-gencode;arch=compute_50,code=sm_50;-gencode;arch=compute_60,code=sm_60;-gencode;arch=compute_61,code=sm_61;-gencode;arch=compute_70,code=sm_70;-gencode;arch=compute_75,code=sm_75;-gencode;arch=compute_80,code=sm_80;-gencode;arch=compute_86,code=sm_86;-gencode;arch=compute_90,code=sm_90;-gencode;arch=compute_37,code=compute_37
  - CuDNN 90.1
  - Magma 2.5.4
  - Build settings: BLAS_INFO=mkl, BUILD_TYPE=Release, COMMIT_SHA=2236df1770800ffea5697b11b0bb0d910b2e59e1, CUDA_VERSION=11.8, CUDNN_VERSION=9.1.0, CXX_COMPILER=C:/actions-runner/_work/pytorch/

In [32]:
torch.set_num_threads(torch.get_num_interop_threads())  # or manually set

In [36]:
if hasattr(core_model, "gradient_checkpointing_disable"):
    core_model.gradient_checkpointing_disable()

if hasattr(core_model, "config"):
    core_model.config.gradient_checkpointing = False

core_model.eval()
scripted = torch.jit.script(core_model)

UnsupportedNodeError: function definitions aren't supported:
  File "C:\Users\tsuma.thomas\Documents\CoreOutline\transformer\ad-hoc-notebooks\..\model_2.py", line 413
    
            if self.config.gradient_checkpointing and self.training:
                def create_custom_forward(module):
                ~~~ <--- HERE
                    def custom_forward(*inputs):
                        return module(*inputs, output_attentions, None)


In [33]:
scripted = torch.jit.script(core_model)
output = scripted(inputs['input_ids'])

UnsupportedNodeError: function definitions aren't supported:
  File "C:\Users\tsuma.thomas\Documents\CoreOutline\transformer\ad-hoc-notebooks\..\model_2.py", line 413
    
            if self.config.gradient_checkpointing and self.training:
                def create_custom_forward(module):
                ~~~ <--- HERE
                    def custom_forward(*inputs):
                        return module(*inputs, output_attentions, None)


In [40]:
core_model = core_model.to(memory_format=torch.channels_last)


In [42]:
!pip install openvino
!pip install openvino-dev[onnx] 

   ---------------------------------------- 0.0/39.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/39.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/39.5 MB 1.3 MB/s eta 0:00:31
   ---------------------------------------- 0.0/39.5 MB 1.3 MB/s eta 0:00:31
   ---------------------------------------- 0.0/39.5 MB 1.3 MB/s eta 0:00:31
   ---------------------------------------- 0.0/39.5 MB 217.9 kB/s eta 0:03:02
   ---------------------------------------- 0.1/39.5 MB 251.0 kB/s eta 0:02:38
   ---------------------------------------- 0.1/39.5 MB 305.0 kB/s eta 0:02:10
   ---------------------------------------- 0.1/39.5 MB 310.3 kB/s eta 0:02:08
   ---------------------------------------- 0.1/39.5 MB 355.0 kB/s eta 0:01:52
   ---------------------------------------- 0.2/39.5 MB 485.3 kB/s eta 0:01:22
   ---------------------------------------- 0.2/39.5 MB 485.3 kB/s eta 0:01:22
   ---------------------------------------- 0.2/39.5 MB 485.3 kB/s eta 0:01:

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\tsuma.thomas\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/37.4 MB 217.9 kB/s eta 0:02:52
   ---------------------------------------- 0.0/37.4 MB 196.9 kB/s eta 0:03:10
   ---------------------------------------- 0.0/37.4 MB 196.9 kB/s eta 0:03:10
   ---------------------------------------- 0.0/37.4 MB 196.9 kB/s eta 0:03:10
   ---------------------------------------- 0.0/37.4 MB 196.9 kB/s eta 0:03:10
   ---------------------------------------- 0.0/37.4 MB 196.9 kB/s eta 0:03:10
   ---------------------------------------- 0.0/37.4 MB 196.9 kB/s eta 0:03:10
   ---------------------------------------- 0.1/37.4 MB 176.6 kB/s eta 0:03:32
   ---------------------------------------- 0.2/37.4 MB 349.3 kB/s eta 0:01:47
   ------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ads 26.0.1 requires protobuf<7.0.0,>=4.25.0, but you have protobuf 3.20.3 which is incompatible.
grpcio-health-checking 1.71.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
grpcio-tools 1.71.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
opentelemetry-proto 1.31.1 requires proto

   - -------------------------------------- 1.3/37.4 MB 62.8 kB/s eta 0:09:36
   - -------------------------------------- 1.3/37.4 MB 62.8 kB/s eta 0:09:36
   - -------------------------------------- 1.3/37.4 MB 62.7 kB/s eta 0:09:37
   - -------------------------------------- 1.3/37.4 MB 62.7 kB/s eta 0:09:37
   - -------------------------------------- 1.3/37.4 MB 62.7 kB/s eta 0:09:37
   - -------------------------------------- 1.3/37.4 MB 62.7 kB/s eta 0:09:37
   - -------------------------------------- 1.3/37.4 MB 62.7 kB/s eta 0:09:37
   - -------------------------------------- 1.3/37.4 MB 62.7 kB/s eta 0:09:37
   - -------------------------------------- 1.3/37.4 MB 63.0 kB/s eta 0:09:34
   - -------------------------------------- 1.3/37.4 MB 63.0 kB/s eta 0:09:34
   - -------------------------------------- 1.3/37.4 MB 63.0 kB/s eta 0:09:34
   - -------------------------------------- 1.3/37.4 MB 63.0 kB/s eta 0:09:34
   - -------------------------------------- 1.3/37.4 MB 62.8 kB/

In [41]:
from openvino.tools.mo import convert_model


ModuleNotFoundError: No module named 'openvino'